# Suite RFL — Reflection

`Reflection` binds the schema data classes to reference object schemas, so that a store's objects are schemas: each
kind of schema and each property, branch or part, parameter and adjacency is an object of its own, linked by
`Schemas.Members` and `Schemas.Types`, and listed by the store's catalog. They are read through the visitor protocols,
written and read as snapshots, validated and built as any objects are.

In [ ]:
from mbse.Schemas.Framework import JSON, Modules, Proxies, Reflection as R, Schemas as S, Validators as V
from support import raises, text

Phone = S.OfObject.Builder().name("Phone").properties(text("number")).create()
Listing = S.OfRelation.Builder().name("Listing").links("directory", "contact").properties(text("key")).unique(
    "contact").create()
Contact = S.OfObject.Builder().name("Contact").ref().description("A person").properties(
    text("name"), lambda p: p.name("home").of(Phone).description("Where to call"),
    lambda p: p.name("tags").of(lambda t: t.as_indexed(lambda i: i.of(lambda x: x.as_native(str)).extent(0, 9)))).relations(
    lambda a: a.name("listings").of(Listing).me("contact")).create()
SCHEMAS = [Contact, Phone, Listing]
n = S.Form.Data("variable", {"name": "n"})
Word = S.OfNative.Builder().name("Word").type(int).parameters(lambda p: p.name("n").of(lambda t: t.as_native(int))).bits(n).create()
Reach = S.OfUnion.Builder().name("Reach").branches(lambda b: b.name("phone").of(Phone), lambda b: b.name("other").of(Phone)).create()
Both = S.OfIntersection.Builder().name("Both").parts(lambda b: b.name("a").of(Phone), lambda b: b.name("b").of(Phone)).create()
Byte = S.OfApply.Builder().name("Byte").of(Word).arguments(8).create()
Keyed = S.OfIndexed.Builder().name("Keyed").key(lambda t: t.as_native(str)).of(Byte).create()
Pair = S.OfRelation.Builder().name("Pair").links("a", "b").create()  # no uniques
Bare = S.OfApply.Builder().name("Bare").of(Word).create()  # no arguments
KINDS = [Word, Reach, Both, Byte, Keyed, Pair, Bare]

## RFL-01 · A store of schemas: its catalog lists them, and every kind and element is an object of its own

In [ ]:
store = R.store(SCHEMAS)
catalog = store.singleton("Schemas.Catalog")
assert catalog.schemas == SCHEMAS and catalog.schema_name() == "Schemas.Catalog" and catalog.owner() is None
assert list(store.extent("Schemas.Object")) == [Contact, Phone] and list(store.extent("Schemas.Relation")) == [Listing]  # the schemas themselves
assert [p.name for p in list(store.extent("Schemas.Property"))] == ["name", "home", "tags", "number", "key"]  # reached through members
assert [a.name for a in list(store.extent("Schemas.Adjacency"))] == ["listings"] and len(list(store.extent("Schemas.Indexed"))) == 1
home = Contact.properties["home"]
assert (Contact.schema_name(), home.schema_name(), Contact.identity() == id(Contact), Contact.owner()) == (
    "Schemas.Object", "Schemas.Property", True, None)
assert [meta.name for meta in R.META.values()] == [
    "Schemas.Native", "Schemas.Object", "Schemas.Union", "Schemas.Intersection", "Schemas.Indexed", "Schemas.Apply",
    "Schemas.Relation", "Schemas.Property", "Schemas.Member", "Schemas.Parameter", "Schemas.Adjacency"]
assert all(meta.ref and meta.validate() == [] for meta in R.META.values())
assert sorted(store.names()) == sorted(["Schemas.Catalog", *(m.name for m in R.META.values()), "Schemas.Members",
                                        "Schemas.Types", "Schemas.Listed"])
assert R.store().singleton("Schemas.Catalog").schemas == []  # an empty catalog
kinds = R.store(KINDS)
assert [s.name for s in list(kinds.extent("Schemas.Native"))] == ["Word", None, None, None]  # and three unnamed: n's, number's, a key's
assert [p.name for p in list(kinds.extent("Schemas.Parameter"))] == ["n"] and [m.name for m in list(kinds.extent("Schemas.Member"))] == [
    "phone", "other", "a", "b"]

## RFL-02 · Read through the visitor protocols: natives, values in their module form, members and types by role

In [ ]:
def members(value):
    return [(e.values["role"], e.values["index"], e.targets["member"]) for e in V.entries_of(value)["members"]]


def types(value):
    return [(e.values["role"], e.targets["type"]) for e in V.entries_of(value)["types"]]


assert V.properties_of(Contact) == {"name": "Contact", "description": "A person", "ref": True}
assert members(Contact) == [("properties", 0, Contact.properties["name"]), ("properties", 1, Contact.properties["home"]),
                            ("properties", 2, Contact.properties["tags"]), ("adjacencies", 0, Contact.adjacencies["listings"])]
home = Contact.properties["home"]
assert V.properties_of(home) == {"name": "home", "description": "Where to call"} and types(home) == [("type", Phone)]
tags = Contact.properties["tags"].type
assert V.properties_of(tags)["extent"].values == {"minimum": 0, "maximum": 9}  # a value, in its module form
assert types(tags) == [("item", tags.item)] and V.properties_of(tags.item) == {"format": "basic", "token": "str"}
assert V.properties_of(Listing)["links"].values == ["directory", "contact"]
assert [u.values for u in V.properties_of(Listing)["uniques"].values] == [["contact"]]
listings = Contact.adjacencies["listings"]
assert V.properties_of(listings) == {"name": "listings", "me": "contact"} and types(listings) == [("relation", Listing)]
assert "bits" not in V.properties_of(Word) and V.properties_of(Word)["terms"].values["bits"].values["kind"] == "variable"
assert members(Word) == [("parameters", 0, Word.parameters["n"])] and types(Word.parameters["n"]) == [("type", Word.parameters["n"].type)]
assert members(Reach) == [("branches", 0, Reach.branches[0]), ("branches", 1, Reach.branches[1])]
assert [m[0] for m in members(Both)] == ["parts", "parts"] and types(Both.parts[0]) == [("type", Phone)]
assert types(Byte) == [("of", Word)] and V.properties_of(Byte)["arguments"].values[0].values["name"] == "n"
assert types(Keyed) == [("item", Byte), ("key", Keyed.key)]
assert "uniques" not in V.properties_of(Pair) and "arguments" not in V.properties_of(Bare) and types(Bare) == [("of", Word)]
assert V.entries_of(Phone).get("types") is None and V.entries_of(Phone)["members"][0].values["role"] == "properties"

## RFL-03 · A snapshot of a store of schemas, read back: schemas whose modules are the same, shared and recursive ones too

In [ ]:
Node = S.OfObject.Builder().name("Node").properties(text("label")).create()
S.OfObject.Builder(Node).properties(lambda p: p.name("next").of(lambda t: t.as_indexed(lambda i: i.of(Node)))).update()
store = R.store([*SCHEMAS, *KINDS, Node])
catalog = store.singleton("Schemas.Catalog")
snapshot = JSON.ToJSON(store).Reachable(R.Catalog.Schema, catalog)
assert snapshot.count('"name": "Phone"') == 1  # shared: written once, by every schema that refers to it
back = JSON.FromJSON(R.store()).Reachable(R.Catalog.Schema, snapshot)
assert [s.name for s in back.schemas] == [s.name for s in catalog.schemas]
assert all(original is not copy for original, copy in zip(catalog.schemas, back.schemas))
assert JSON.ToJSON(R.store()).Reachable(R.Catalog.Schema, back) == snapshot
modules = Proxies.OfStore()
assert JSON.ToJSON(modules)(S.Module.Schema, Modules.module(modules, back.schemas)) == JSON.ToJSON(modules)(
    S.Module.Schema, Modules.module(modules, catalog.schemas))  # the same schemas
copy = back.schemas[-1]
assert copy.properties["next"].type.item is copy  # recursive: the node's list holds nodes
assert back.schemas[0].properties["home"].type is back.schemas[1]  # shared: one Phone
assert back.schemas[6].arguments == {"n": 8} and back.schemas[6].of is back.schemas[3]  # Byte applies Word
assert back.schemas[3].bits == n and back.schemas[3].parameters["n"].type == S.OfNative.Data(int)

## RFL-04 · Validated, built and updated as any objects; a native another format names cannot be read

In [ ]:
store = R.store([*SCHEMAS, *KINDS])
assert V.Validate(store).Reachable(R.Catalog.Schema, store.singleton("Schemas.Catalog")) == []
home = Contact.properties["home"]
builder = store.builder("Schemas.Property", home)
builder.property("description", lambda p: p.value(lambda a: a.as_native(lambda v: v.set("The home phone"))))
assert builder.update() is home and home.description == "The home phone" and home.type is Phone  # the schema itself
fresh = store.builder("Schemas.Property")
fresh.property("name", lambda p: p.value(lambda a: a.as_native(lambda v: v.set("fax"))))
made = fresh.create()
assert isinstance(made, S.OfProperty.Data) and (made.name, made.type) == ("fax", None)
assert store.builder("Schemas.Catalog", store.singleton("Schemas.Catalog")).update().schemas == [*SCHEMAS, *KINDS]
foreign = S.OfNative.resolve(lambda t: t.token("ccpp", "int32_t"))
assert list(R.store([foreign]).extent("Schemas.Native")) == [foreign]
with raises(TypeError, match="unsupported native type 'x'"):
    V.properties_of(S.OfNative.Data("x"))